In [3]:
# -*- coding: utf-8 -*-
"""Shougou.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1YnryZhOsRT7X6cAsNZBrP4vgBIIiJkZJ
"""

#from IPython.utils.io import atomic_writing
from openpyxl.styles import Border, Side
from openpyxl.utils import get_column_letter
from openpyxl.styles import PatternFill
import openpyxl
import pandas as pd
import re


wb = openpyxl.Workbook()
output = wb.active
#アクティブなシート名を"diff"にする。
output.title = "diff"


#csvファイルの読み込みをする。文字コードShift-JISを指定する。
#record = pd.read_csv(r"C:\Users\fg5ve\OneDrive\Desktop\Python練習問題\Shougou\看護記録書Ⅱ_2026年07月01日2026年07月31日.edit.csv", encoding="cp932")
#actual = pd.read_csv(r"C:\Users\fg5ve\OneDrive\Desktop\Python練習問題\Shougou\訪問看護スケジュール_202607_実績.edit.csv", encoding="cp932")
record = pd.read_csv("看護記録書Ⅱ_2026年07月01日2026年07月31日.edit.csv", encoding="cp932")
actual = pd.read_csv("訪問看護スケジュール_202607_実績.edit.csv", encoding="cp932")


######"訪問看護スケジュール_202607_実績.edit.csv"の中身をディクショナリに格納する部分######
f_name_actual = "訪問看護スケジュール_202607_実績.edit.csv"
#f_name_actualの文字列の中から数字が6個連続している部分を探し、
#結果を変数matchへ入れる（見つかればMatchオブジェクト、見つからなければNone）。
#もしmatchがNoneでなければ、matchオブジェクトから.group()で一致した文字列を取り出し、
#yyyymmに入れて表示する。
match = re.search(r"\d{6}", f_name_actual)
if match:
    yyyymm = match.group()

year_month = yyyymm[:4] + "-" + yyyymm[4:]

#訪問看護スケジュールのcsvの日付を格納するディクショナリact_data
#取得した日付の前に"2026-07-"を付けて格納するディクショナリact_YYYYMMDD
act_date = {}
act_YYYYMMDD = {}
act_Shokuin1 = {}
act_Riyousha = {}
act_Start_time = {}
act_End_time = {}
act_dict = {}

#訪問看護スケジュールのcsvの最初から文字列が入っている最後までiをインクリメントする
for i in range(len(actual)):
    #locは列名でi行目の中身を元のデータ型のまま取得し、act_data[i]に格納する
    act_date[i] = actual.loc[i, "日付"]

    #日付を2桁表示にする
    act_date[i] = f"{act_date[i]:02d}"

    #20YY-MM-DD形式に変換した文字列をact_YYYYMMDDへ格納する
    act_YYYYMMDD[i] = year_month + "-" + act_date[i]

    act_Shokuin1[i] = actual.loc[i, "職員名１"]
    act_Riyousha[i] = actual.loc[i, "利用者"]
    act_Start_time[i] = actual.loc[i, "開始時間"]
    act_End_time[i] = actual.loc[i, "終了時間"]

    #ディクショナリact_dictのキーは("日付", "利用者", "開始時間")
    #値は("職員名１", "終了時間)"
    key = (act_YYYYMMDD[i], act_Riyousha[i], act_Start_time[i])
    act_dict[key] ={"職員名１": act_Shokuin1[i], "終了時間": act_End_time[i],}
for key in act_dict: act_dict[key]["職員名"] = act_dict[key].pop("職員名１")


######"看護記録書Ⅱ_2026年07月01日2026年07月31日.edit.csv"の中身をディクショナリに格納する部分######

rec_date = {}
rec_YYYYMMDD = {}
rec_Shokuin1 = {}
rec_Riyousha = {}
rec_Start_time = {}
rec_End_time = {}
rec_dict = {}

#看護記録書Ⅱのcsvの最初から文字列が入っている最後までiをインクリメントする
for i in range(len(record)):
    rec_date[i] = record.loc[i, "訪問日"]
    rec_Shokuin1[i] = record.loc[i, "職員名1"]
    rec_Riyousha[i] = record.loc[i, "利用者名"]
    rec_Start_time[i] = record.loc[i, "訪問開始時刻"]
    rec_End_time[i] = record.loc[i, "訪問終了時刻"]

    #ディクショナリrec_dictのキーは("日付", "利用者", "開始時間")
    #値は("職員名1", "訪問終了時刻)"
    key = (rec_date[i], rec_Riyousha[i], rec_Start_time[i])
    rec_dict[key] ={"職員名1": rec_Shokuin1[i], "訪問終了時刻": rec_End_time[i]}
for key in rec_dict:
    rec_dict[key]["職員名"] = rec_dict[key].pop("職員名1")
    rec_dict[key]["終了時間"] = rec_dict[key].pop("訪問終了時刻")


######act_dictとrec_dictの中身を比較する######
row = 2

output["A1"] = "参照元"
output["B1"] = "日付"
output["C1"] = "職員名"
output["D1"] = "利用者名"
output["E1"] = "開始時間"
output["F1"] = "終了時間"

orange_fill = PatternFill(start_color="FFA500", end_color="FFA500", fill_type="solid")
blue_fill = PatternFill(start_color="ADD8E6", end_color="ADD8E6", fill_type="solid")

for key in act_dict:
    if key in rec_dict:
        #①看護記録Ⅱと訪問看護スケジュール実績の"職員名"または"終了時間"が異なった場合、output.xlsxに相違を出力する
        if act_dict[key]["職員名"] != rec_dict[key]["職員名"] or act_dict[key]["終了時間"] != rec_dict[key]["終了時間"]:
              #A列に"看護記録"と入力する
              output[f"A{row}"] = "看護記録"
              #B-F列にB:"日付", C:"職員名", D:"利用者名", E:"開始時間", F:"終了時間"を入力する
              output[f"B{row}"] = key[0]
              output[f"C{row}"] = rec_dict[key]["職員名"]
              output[f"D{row}"] = key[1]
              output[f"E{row}"] = key[2]
              output[f"F{row}"] = rec_dict[key]["終了時間"]

              #違う項目のセルだけオレンジにする(看護記録行)
              if act_dict[key]["職員名"] != rec_dict[key]["職員名"]:
                  output[f"C{row}"].fill = orange_fill
              if act_dict[key]["終了時間"] != rec_dict[key]["終了時間"]:
                  output[f"F{row}"].fill = orange_fill

              #1行下に行く
              row += 1

              #A列に"訪看実績"と入力する
              output[f"A{row}"] = "訪看実績"
              #B-F列にB:"日付", C:"職員名", D:"利用者名", E:"開始時間", F:"終了時間"を入力する
              output[f"B{row}"] = key[0]
              output[f"C{row}"] = act_dict[key]["職員名"]
              output[f"D{row}"] = key[1]
              output[f"E{row}"] = key[2]
              output[f"F{row}"] = act_dict[key]["終了時間"]

              #違う項目のセルだけオレンジにする(訪看実績行)
              if act_dict[key]["職員名"] != rec_dict[key]["職員名"]:
                  output[f"C{row}"].fill = orange_fill
              if act_dict[key]["終了時間"] != rec_dict[key]["終了時間"]:
                  output[f"F{row}"].fill = orange_fill

              row += 1

    else:
        # ② 訪問看護スケジュール実績にはあるが、看護記録Ⅱには存在しない
        #    看護記録側の行はラベルのみ、B〜F列は空白のまま(背景を水色にする)
        output[f"A{row}"] = "看護記録"
        for col in "BCDEF":
            output[f"{col}{row}"].fill = blue_fill
        row += 1

        #A列に"訪看実績"と入力する
        output[f"A{row}"] = "訪看実績"
        #B-F列にB:"日付", C:"職員名", D:"利用者名", E:"開始時間", F:"終了時間"を入力する
        output[f"B{row}"] = key[0]
        output[f"C{row}"] = act_dict[key]["職員名"]
        output[f"D{row}"] = key[1]
        output[f"E{row}"] = key[2]
        output[f"F{row}"] = act_dict[key]["終了時間"]
        row += 1

# ③ 看護記録Ⅱにはあるが、訪問看護スケジュール実績には存在しない
for key in rec_dict:
    if key not in act_dict:
        output[f"A{row}"] = "看護記録"
        output[f"B{row}"] = key[0]
        output[f"C{row}"] = rec_dict[key]["職員名"]
        output[f"D{row}"] = key[1]
        output[f"E{row}"] = key[2]
        output[f"F{row}"] = rec_dict[key]["終了時間"]
        row += 1

        output[f"A{row}"] = "訪看実績"
        #    訪看実績側の行はラベルのみ、B〜F列は空白のまま(背景を水色にする)
        for col in "BCDEF":
            output[f"{col}{row}"].fill = blue_fill
        row += 1


###A-F列の列幅をすべて13に統一する
for col_idx in range(1, 7):  # A列(1)〜F列(6)
    col_letter = get_column_letter(col_idx)
    output.column_dimensions[col_letter].width = 13

###罫線を引く処理###
thin = Side(style='thin', color='000000')
thick = Side(style='thick', color='000000')

max_row = output.max_row
max_col = 6

# ① まず全体に細い罫線を引く(この段階では1行目も含めて全部細線でOK)
for row in output.iter_rows(min_row=1, max_row=max_row, min_col=1, max_col=max_col):
    for cell in row:
        cell.border = Border(left=thin, right=thin, top=thin, bottom=thin)

# ② 2行ごとの区切りに太線(該当行の上側)
thick_border_rows = list(range(4, max_row + 1, 2))
for row_num in thick_border_rows:
    for col_num in range(1, max_col + 1):
        cell = output.cell(row=row_num, column=col_num)
        cell.border = Border(
            left=cell.border.left,
            right=cell.border.right,
            top=thick,
            bottom=cell.border.bottom
        )

# ③ 1行目(ラベル行)の下に太線
for col_num in range(1, max_col + 1):
    cell = output.cell(row=1, column=col_num)
    cell.border = Border(
        left=cell.border.left,
        right=cell.border.right,
        top=cell.border.top,
        bottom=thick
    )

# ④ 最終行の下に太線
for col_num in range(1, max_col + 1):
    cell = output.cell(row=max_row, column=col_num)
    cell.border = Border(
        left=cell.border.left,
        right=cell.border.right,
        top=cell.border.top,
        bottom=thick
    )

#相違情報を入力したエクセルをRESULTS.xlsxとして指定したパスに作成する。
#wb.save(r"C:\Users\fg5ve\OneDrive\Desktop\Python練習問題\Shougou\RESULTS.xlsx")
wb.save("RESULTS.xlsx")

"""
for r in range(1, 10):
      for c in "ABCDEF":
            print(f"{c}{r}:", output[f"{c}{r}"].value, end="  ")
      print()
"""





'\nfor r in range(1, 10):\n      for c in "ABCDEF":\n            print(f"{c}{r}:", output[f"{c}{r}"].value, end="  ")\n      print()\n'